# eVTOL Multidisciplinary Design Optimization (MDO)

**Single Objective Optimization**

___Note on Binder Stability: JAX requires an initial "warm-up" period to compile the physics equations into machine code. If the kernel restarts on the first attempt, please run the cells again; subsequent runs will be significantly faster due to the persistent JAX compilation cache.___

---

## 1. Environment Setup

In [10]:
# Imports
import os
import sys
import time
import numpy as np
import src.parameters as parameters
from src.optimizer.eVTOL_group import eVTOLGroup
from src.analysis.evaluation_model import full_model_evaluation
import importlib
import src.analysis.evaluation_model as evaluation_model
from src.analysis.evaluation_model import write_results_to_excel, display_model_dashboard

# Binder-friendly JAX settings - JAX allows for automatic differentiation, just-in-time compilation, and GPU/TPU acceleration
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['JAX_ENABLE_X64'] = '1'

# Ensure local imports work in cloud notebooks
sys.path.append(os.getcwd())

import jax
import openmdao.api as om

# persistent jax compilation cache
ROOT = os.getcwd()
jax.config.update('jax_compilation_cache_dir', os.path.join(ROOT, 'jax_cache'))
try:
    jax.config.update('jax_enable_x64', True)
except Exception:
    pass



print('Environment and imports ready')

Environment and imports ready


---

## 2. OpenMDAO Optimizaiton Problem setup (building the problem)

In [11]:
# OpenMDAO Problem setup (build the problem)
prob = om.Problem(model=eVTOLGroup(parameters=parameters))

print('eVTOLGroup model instantiated and added to problem.')

eVTOLGroup model instantiated and added to problem.


### 2.1 Design Variables (define the design space)

In [12]:
# Design Variables (Independent variables) - creating a "component" that holds our design variables
# what the optimizer is allowed to touch to find the best design
# We provide "placeholder" initial values just to define the variables (they will be overwritten in the opt intializaiton by x(0))
iv = om.IndepVarComp()
iv.add_output('b', val=15.0) 
iv.add_output('c', val=1.0)
iv.add_output('r_cruise', val=1.0)
iv.add_output('r_hover', val=1.0)
iv.add_output('rho_bat', val=400.0)
iv.add_output('c_charge', val=2.0)
prob.model.add_subsystem('iv', iv, promotes=['*'])

# Design Variables bounds (and set scaling/pre-conditioning)
# ref: normalize (scaling) variables of different magnitudes (e.g., 1.5m vs 400 Wh/kg) to a 0 to 1 scale for smooth opt
prob.model.add_design_var('b', lower=6.0, upper=15.0, ref0=6.0, ref=15.0)
prob.model.add_design_var('c', lower=1.0, upper=2.5, ref0=1.0, ref=2.5)
prob.model.add_design_var('r_cruise', lower=0.6, upper=2.5, ref0=0.6, ref=2.5)
prob.model.add_design_var('r_hover', lower=0.6, upper=2.0, ref0=0.6, ref=2.0)
prob.model.add_design_var('rho_bat', lower=200.0, upper=400.0, ref0=200.0, ref=400.0)
prob.model.add_design_var('c_charge', lower=1.0, upper=4.0, ref0=1.0, ref=4.0)

print('Design variables set and valid.')

Design variables set and valid.


### 2.2 Constraints (setting boundaries)

In [13]:
# Constraints: Setting the constraints - creating a "component" that holds our constraints in OpenMDAO

# computation for the constraints component with initial values (these will be overwritten by the optimizer)
cons_comp = om.ExecComp('c1 = b - rotor_spacing',
                       b=15.0,
                       rotor_spacing=1.0)
prob.model.add_subsystem('cons_comp', cons_comp)

# connecting the constraint with the model
prob.model.connect('rotor_spacing', 'cons_comp.rotor_spacing')
prob.model.connect('b', 'cons_comp.b')

# The Enforcer: Constraints and design vars (scaling for the margins of the constraints)
prob.model.add_constraint('cons_comp.c1', lower=0.0, ref=1.0)
prob.model.add_constraint('vertiport_span', upper=15.0, ref=1.0)
prob.model.add_constraint('MTOM', upper=5700.0, ref=2000.0)
prob.model.add_constraint('V_cruise', upper=129.0, ref=100.0)
prob.model.add_constraint('V_climb', upper=129.0, ref=100.0)

print('Constraints set.')

Constraints set.


### 2.3 Objective (defining the goal)

In [14]:
# Optimization objective:
# We scale the objective to order 1 so the optimiser operates smoothly
# ref = estimate on the order of magnitude of the objective to keep the optimization well-scaled
# negative ref (ref = -x) to maximize, positive ref (ref = x) to minimize

prob.model.add_objective('GWP_flight', ref=20)

print('Objective set and valid.')

Objective set and valid.


### 2.4 Optimizaiton Driver (setting up the engine)

In [15]:
# Driver setup (SLSQP) (Sequential Least Squares Programming) 
# gradient-based optimization algorithm that handles nonlinear problems with constraints effectively
# tol = tolerance for convergence (stop condition)
# disp = display optimization output, 
# declare_coloring = tells OpenMDAO to analyze which variables don't affect each other so it can calculate derivatives in parallel 
#   (enable sparsity pattern for efficient derivatives)

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'SLSQP'
prob.driver.options['tol'] = 1e-6
prob.driver.options['disp'] = True
prob.driver.declare_coloring() if hasattr(prob.driver, 'declare_coloring') else None

# Initialization
# Gradient-based opt is iterative, 
# they need a starting point in the "design space" to begin searching for the optimum.
# we use the middle of the design variable bounds as a reasonable starting point for the optimization

x0 = np.array([10.5, 1.75, 1.55, 1.3, 350.0, 3.0], dtype=float)

# Finalize setup and initialize values
# prob.setup(): checks that all 'connect' statements actually go to real variables 
# - if made a typo in a variable name, this is where the code will crash
# prob.set_val(): initializes the values of the design variables to the starting point (x0) for the optimization

prob.setup()
prob.set_val('b', x0[0])
prob.set_val('c', x0[1])
prob.set_val('r_cruise', x0[2])
prob.set_val('r_hover', x0[3])
prob.set_val('rho_bat', x0[4])
prob.set_val('c_charge', x0[5])

print('Problem built and ready.')

Problem built and ready.


---

## 3. Run the Optimization

In [16]:
# stopwatch for optimization time
t0 = time.time()

# prob.run_driver(): runs the optimization driver, which iteratively calls the model to evaluate the objective and constraints,
# and uses the results to update the design variables until convergence is achieved or an error occurs.

try:
    prob.run_driver()
except Exception as exc:
    print('Optimization failed:', exc)
    raise
t1 = time.time()
print(f'Elapsed (s): {t1-t0:.1f}')

+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 0 ; 1742.58691 1
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 1 ; 1246.90609 0.715548864


/Users/johannesjanning/Library/CloudStorage/Dropbox/Mac (3)/Documents/VSC/eVTOL_OpenMDAO_v3_notebook_building/.venv/lib/python3.11/site-packages/openmdao/utils/relevance.py:1234: OpenMDAOWarning:The top level group has a nonlinear solver that computes gradients, so the entire model will be included in the optimization iteration.


+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 2 ; 0.116247508 6.67097333e-05
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 3 ; 8.09253387e-08 4.64397718e-11
NL: Newton Converged
No coloring was computed successfully.
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 1 iterations
NL: Newton 0 ; 6.93582802e-09 1
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 1 ; 9.44353799e-13 0.000136155885
NL: Newton Converged


/Users/johannesjanning/Library/CloudStorage/Dropbox/Mac (3)/Documents/VSC/eVTOL_OpenMDAO_v3_notebook_building/.venv/lib/python3.11/site-packages/openmdao/utils/coloring.py:428: DerivativesWarning:Coloring was deactivated.  Improvement of 0.0% was less than min allowed (5.0%).


+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 0 ; 389.106178 1
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 1 ; 118.322396 0.304087683
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 2 ; 0.0778716396 0.000200129538
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 3 ; 8.15372819e-10 2.09550211e-12
NL: Newton Converged
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 0 ; 5.4151106 1
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 1 ; 0.0035651558 0.000658371741
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 2 ; 6.37749016e-09 1.17772113e-09
NL: Newton Converged
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 0 ; 7.6397058 1
+  
+  ====
+  mtom
+  ====
+  NL: Newton Converged in 0 iterations
NL: Newton 1 ; 0.00475569891 0.000622497651
+  
+ 

---

## 4. Data Extraction

In [17]:
# - sval: A helper to safely extract scalar values from the OpenMDAO solver.
# - b_opt...c_charge_opt: Capturing the "Winning Design" into standard variables.
# - Print: Displaying the final Design Vector for the optimized eVTOL

def sval(name):
    try:
        return float(np.asarray(prob.get_val(name)).item())
    except Exception:
        return None

b_opt = sval('b')
c_opt = sval('c')
r_cruise_opt = sval('r_cruise')
r_hover_opt = sval('r_hover')
rho_bat_opt = sval('rho_bat')
c_charge_opt = sval('c_charge')

print('Optimal design vector:')
print([b_opt, c_opt, r_cruise_opt, r_hover_opt, rho_bat_opt, c_charge_opt])

Optimal design vector:
[15.0, 1.0, 1.352991989982466, 1.6773504320663268, 400.0, 1.0000000000000013]


---

## 5. Design Results

In [18]:
# 1. full_model_evaluation: Generates a deep-dive dataset for the winning design.
# 2. baseline_vector: Used to calculate "Optimization Gain" (Best vs. Standard).
# 3. Excel & Dashboard: Exports results for stakeholders and displays key performance metrics.

# post-optimization evaluation to get the full set of performance metrics for the optimized design
model_results, comparison_table = full_model_evaluation(
    b_opt, c_opt, r_cruise_opt, r_hover_opt, rho_bat_opt, c_charge_opt, parameters
)

# Define your baseline for the comparison logic
baseline_vector = [15.0, 1.0, 1.352991989982466, 1.6773504320663268, 400.0, 1.0000000000000013]

# Generates a detailed Excel report containing the full model
importlib.reload(evaluation_model)
out_path, df_results = write_results_to_excel(
    results_dict=model_results,
    comparison_list=comparison_table,
    mode='GWP',
    filename='optimized_results_GWP.xlsx',
    baseline=baseline_vector
)

# visual dashboard for the optimized design
print("-" * 30)
print("  OPTIMIZED DESIGN DASHBOARD")
print("-" * 30)
# Make sure display_model_dashboard is updated to handle the 'baseline' argument
display_model_dashboard(model_results, baseline=baseline_vector)

Results written to: src/results/GWP/optimized_results_GWP.xlsx
------------------------------
  OPTIMIZED DESIGN DASHBOARD
------------------------------


Section,Metric,Value,Baseline,Unit,% Change
DESIGN VARIABLE,Wing span,15.000,15.000,m,+0.00%
DESIGN VARIABLE,Chord length,1.000,1.000,m,+0.00%
DESIGN VARIABLE,Pusher rotor radius,1.353,1.353,m,+0.00%
DESIGN VARIABLE,Hover rotor radius,1.677,1.677,m,+0.00%
DESIGN VARIABLE,Battery energy density,400.000,400.000,Wh/kg,+0.00%
DESIGN VARIABLE,Charging C-rate,1.000,1.000,1/h,+0.00%
AERODYNAMICS,Wing Aspect Ratio,15.000,15.000,-,+0.00%
AERODYNAMICS,Lift coefficient (cruise),0.545,0.545,-,+0.00%
AERODYNAMICS,Drag coefficient (cruise),0.048,0.048,-,+0.00%
AERODYNAMICS,Lift-to-drag Ratio (cruise),11.448,11.448,-,+0.00%
